# Post-hoc study - train on every slide of a patient (development CV only)

The headline result is final (test set used once). This notebook embeds the **144 further
slides of the 162 development patients** (same CLWD release; test patients' slides are never
downloaded) and asks, on the same 5 development folds, whether training on them helps.
See `scripts/09_extra_slides_study.py`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, Secret `HF_TOKEN`, input dataset
`clwd-embeddings` (the stage-1 output). Use **Save & Run All (Commit)**: about 3 h to embed
the slides, then about 1-2 h for the study.

In [ ]:
# Clone the pinned branch of the repository and install only what Kaggle lacks.
# NOTE: no "pip -U": upgrading huggingface_hub to 2.x breaks Kaggle's transformers.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/pathology-fusion-AI"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", "https://github.com/jkaral/pathology-fusion-AI.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements-kaggle.txt
!git log -1 --format="commit %h %s"

In [ ]:
# Secrets live in Kaggle "Add-ons -> Secrets" (never in the repo or notebook).
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[name] = s.get_secret(name)
            print(f"{name}: loaded")
        except Exception:
            print(f"{name}: not set")
except ImportError:
    print("Not on Kaggle: set HF_TOKEN / WANDB_API_KEY in the environment yourself.")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Embed the extra slides (resumable; skips finished slides)

In [ ]:
import shutil
os.environ["PATHFUSION_JPG_CACHE"] = "/tmp"
OUT_X = "/kaggle/working/embeddings_extra"
print("free GB in /tmp:", shutil.disk_usage("/tmp").free / 1e9)
!python scripts/01_extract_embeddings.py --extra-slides --out-dir {OUT_X} paths.qc_dir=/kaggle/working/qc_extra
!python scripts/01_extract_embeddings.py --summarize --out-dir {OUT_X}

## 2. One directory with headline + extra features (symlinks)

In [ ]:
import glob
EMB = "/kaggle/working/all_embeddings"
os.makedirs(EMB, exist_ok=True)
srcs = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/WSI-*.h5", recursive=True)}) + [OUT_X]
for d in srcs:
    for p in glob.glob(f"{d}/WSI-*.h5"):
        dst = os.path.join(EMB, os.path.basename(p))
        if not os.path.exists(dst):
            os.symlink(p, dst)
print(len(glob.glob(f"{EMB}/*.h5")), "feature files (expected 203 headline + 144 extra = 347)")

## 3. The study (development CV only; the test set is never loaded)

In [ ]:
!python scripts/09_extra_slides_study.py --embeddings-dir {EMB} --seeds 0,1,2
!cat artifacts/extra_slides_study/results.md

In [ ]:
!cp {OUT_X}/extraction_summary.json artifacts/extra_slides_study/extraction_summary_extra.json
!tar czf /kaggle/working/extra_slides_bundle.tgz artifacts/extra_slides_study
print("Download /kaggle/working/extra_slides_bundle.tgz from the Output tab")